<a href="https://colab.research.google.com/github/SyameimaruKoa/wd14-tagger-xmp/blob/main/run_colab_server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WD14 Tagger Universal - Google Colab サーバー

このノートブックは、Google Colab 上で WD14 Tagger をサーバーモードとして起動し、ローカル環境から接続して画像タグ付けを行うためのものじゃ。
接続のセキュリティ確保と簡便化のために **Tailscale** を使用するぞ。

## 事前準備（Tailscale 認証キーの作成と Colab への登録）

### 1. Tailscale の認証キー（Auth Key）作成
1. Tailscale の管理画面の [Settings > Keys](https://login.tailscale.com/admin/settings/keys) にアクセスするのじゃ。
2. **Generate auth key** ボタンをクリックするのじゃ。
3. 設定項目を以下のように構成するのじゃ：
   - **Description**: 分かりやすい名前を入力（例: `colab-wd14-server`）
   - **Reusable**: チェックを入れる（ON）。これにより、Colab インスタンスの再起動時にも同じキーを使い回せるぞ。
   - **Ephemeral**: チェックを入れる（ON）。これをしておくと、Colab インスタンスがシャットダウンしてオフラインになった際に、Tailscale 側で自動的にデバイス（ノード）が削除（切断）されるため、ゴミが残らず綺麗じゃ。
   - **Tags**: `tag:Guest` を選択するのじゃ。
     - ※もし `tag:Guest` が選択肢に表示されない場合は、事前に Tailscale の Access Control Policy (ACL) にて `tagOwners` などで `tag:Guest` を定義しておく必要があるぞ。
4. **Generate key** をクリックし、生成された `tskey-auth-...` で始まるキーをコピーするのじゃ。

### 2. Google Colab へのシークレットの登録
1. Left sidebarにある **鍵マーク（Secrets / シークレット）** アイコンをクリックするのじゃ。
2. **Add new secret** をクリックして以下を入力するのじゃ：
   - Name: `TAILSCALE_AUTHKEY`
   - Value: 先ほどコピーした Tailscale の認証キー
3. 追加したシークレットの **Notebook access (ノートブック of access)** トグルスイッチを **ON** にするのじゃ（これでコードからシークレットを読み込めるようになるぞ）。

※もしシークレットを登録していない（または登録したくない）場合は、セル実行時に入力プロンプトが表示されるので、そこにコピーしたキー、または Tailscale 画面に表示されるインストール/セットアップ用コマンド（`curl ... && sudo tailscale up --auth-key=tskey-auth-xxxx`）をそのまま貼り付ければ、キー部分を自動抽出して使用できるぞ。

## 1. Environment構築 & リポジトリの準備

プロジェクトのファイルをクローンし、必要なライブラリおよびONNX Runtime（GPUが利用可能ならシステムCUDAバージョンに応じたGPU版、無ければCPU版）をインストールするのじゃ。

In [ ]:
import os, subprocess, re, sys
has_gpu = False
try:
    subprocess.run("nvidia-smi", stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, shell=True, check=True)
    has_gpu = True
except Exception:
    pass
ort_pkg = "onnxruntime"
if has_gpu:
    cuda_major = 12
    try:
        res = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
        match = re.search(r"release (\d+)\.", res.stdout)
        if match: cuda_major = int(match.group(1))
    except Exception:
        if os.path.exists("/usr/local/cuda"):
            try:
                real_path = os.path.realpath("/usr/local/cuda")
                match = re.search(r"cuda-(\d+)\.", real_path)
                if match: cuda_major = int(match.group(1))
            except Exception: pass
    if cuda_major >= 13:
        ort_pkg = "onnxruntime-gpu"
    elif cuda_major == 12:
        ort_pkg = "onnxruntime-gpu<1.27.0"
    else:
        ort_pkg = "onnxruntime-gpu<1.17.0"
if not os.path.exists("embed_tags_universal.py"):
    !git clone https://github.com/SyameimaruKoa/wd14-tagger-xmp.git
    %cd wd14-tagger-xmp
else:
    !git pull
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", ort_pkg], check=True)

## 2. Tailscale のインストールと接続

Tailscale をインストールして起動し、`TAILSCALE_AUTHKEY` を用いてログインするのじゃ。ホスト名は `wd14-tagger-colab` で固定されるぞ。

In [ ]:
import os, subprocess, re, time
auth = None
try:
    from google.colab import userdata
    auth = userdata.get("TAILSCALE_AUTHKEY")
except Exception:
    pass
if not auth:
    inp = input("Tailscale Auth Key (or setup command): ").strip()
    match = re.search(r"(tskey-(?:auth-)?[a-zA-Z0-9_-]+)", inp)
    auth = match.group(1) if match else inp
if not auth:
    raise ValueError("Tailscale Auth Key is required.")
!curl -fsSL https://tailscale.com/install.sh | sh
!mkdir -p /var/run/tailscale /var/lib/tailscale
subprocess.Popen(["tailscaled", "--tun=userspace-networking"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(20):
    if os.path.exists("/var/run/tailscale/tailscaled.sock"): break
    time.sleep(0.5)
res = subprocess.run(["tailscale", "up", f"--auth-key={auth}", "--hostname=wd14-tagger-colab", "--accept-dns=false", "--ssh", "--reset"], capture_output=True, text=True)
if res.returncode != 0:
    print(f"Error: {res.stderr.strip()}")
    res.check_returncode()
subprocess.run(["tailscale", "serve", "--bg", "--tcp", "5000", "5000"], capture_output=True)

## 3. 推論サーバーの起動

推論サーバーをポート 5000 で起動するのじゃ。下に表示されるダッシュボードの指示に従ってローカルPCから接続するのじゃ。
サーバーの動作を終了し、Tailscaleを切断する場合は、セルの左側にある停止ボタン（■）を押すのじゃ。

In [ ]:
import os, sys, subprocess, site, json, asyncio
from IPython.display import display, HTML
ip = subprocess.run(["tailscale", "ip", "-4"], capture_output=True, text=True).stdout.strip()
hostname = "wd14-tagger-colab"
try:
    status_raw = subprocess.run(["tailscale", "status", "--json"], capture_output=True, text=True).stdout
    status = json.loads(status_raw)
    hostname = status.get("Self", {}).get("HostName", hostname)
except Exception:
    pass
html_code = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; background: #1e1e2e; color: #cdd6f4; padding: 20px; border-radius: 12px; max-width: 700px; box-shadow: 0 4px 20px rgba(0,0,0,0.15); margin: 15px 0; border: 1px solid #313244;">
  <div style="display: flex; align-items: center; justify-content: space-between; border-bottom: 1px solid #313244; padding-bottom: 12px; margin-bottom: 16px;">
    <div style="display: flex; align-items: center;">
      <span style="display: inline-block; width: 10px; height: 10px; background-color: #a6e3a1; border-radius: 50%; margin-right: 8px; box-shadow: 0 0 8px #a6e3a1; animation: pulse 1.8s infinite;"></span>
      <span style="font-size: 16px; font-weight: 600; color: #a6e3a1;">WD14 Tagger Server Active</span>
    </div>
    <span style="font-size: 11px; background: #313244; color: #cdd6f4; padding: 3px 8px; border-radius: 4px; font-weight: 500;">Port: 5000</span>
  </div>
  <div style="background: rgba(249, 226, 175, 0.1); border: 1px solid #f9e2af; padding: 12px; border-radius: 8px; margin-bottom: 16px; font-size: 12px; color: #f9e2af; line-height: 1.5;">
    ⚠️ <strong>注意:</strong> 起動後、初回接続時は Hugging Face からモデルファイルを自動ダウンロードするため、ロード完了まで 1〜2分 ほどかかります。下に「<code>[INFO] サーバー稼働中 Port: 5000</code>」と表示されるまで、ローカルからの接続はお待ちください。
  </div>
  <div style="display: flex; gap: 12px; margin-bottom: 16px;">
    <div onclick="navigator.clipboard.writeText('{ip}'); alert('Copied Tailscale IP to clipboard: {ip}');" style="flex: 1; background: #11111b; padding: 12px; border-radius: 8px; border: 1px solid #313244; cursor: pointer; position: relative; transition: all 0.2s;" onmouseover="this.style.borderColor='#89b4fa'; this.style.background='#181825';" onmouseout="this.style.borderColor='#313244'; this.style.background='#11111b';">
      <div style="font-size: 10px; color: #a6adc8; text-transform: uppercase; margin-bottom: 4px; font-weight: bold; display: flex; justify-content: space-between;">
        <span>Tailscale IP</span>
        <span style="color: #89b4fa; font-size: 9px;">Click to Copy</span>
      </div>
      <div style="font-family: monospace; font-size: 14px; color: #89b4fa; word-break: break-all;">{ip}</div>
    </div>
    <div onclick="navigator.clipboard.writeText('{hostname}'); alert('Copied MagicDNS Hostname to clipboard: {hostname}');" style="flex: 1; background: #11111b; padding: 12px; border-radius: 8px; border: 1px solid #313244; cursor: pointer; position: relative; transition: all 0.2s;" onmouseover="this.style.borderColor='#f9e2af'; this.style.background='#181825';" onmouseout="this.style.borderColor='#313244'; this.style.background='#11111b';">
      <div style="font-size: 10px; color: #a6adc8; text-transform: uppercase; margin-bottom: 4px; font-weight: bold; display: flex; justify-content: space-between;">
        <span>MagicDNS Hostname</span>
        <span style="color: #f9e2af; font-size: 9px;">Click to Copy</span>
      </div>
      <div style="font-family: monospace; font-size: 14px; color: #f9e2af; word-break: break-all;">{hostname}</div>
    </div>
  </div>
  <div style="margin-bottom: 16px;">
    <h4 style="margin: 0 0 10px 0; font-size: 13px; color: #bac2de; font-weight: 600;">Connection Instructions:</h4>
    <div style="margin-bottom: 12px; background: #181825; padding: 12px; border-radius: 8px; border: 1px solid #313244;">
      <div style="font-size: 12px; color: #89b4fa; font-weight: bold; margin-bottom: 6px;">PowerShell (Windows)</div>
      <div style="font-size: 11px; color: #a6e3a1; margin-bottom: 4px;">● MagicDNS (Recommended)</div>
      <pre style="background: #11111b; padding: 8px; border-radius: 6px; font-size: 11px; font-family: monospace; overflow-x: auto; margin: 0 0 10px 0; color: #cdd6f4; border: 1px solid #313244; white-space: pre-wrap; word-break: break-all;">.\\run_tagger.ps1 -Client -HostIP \"{hostname}\" -Path \"C:\\Images\" -Organize</pre>
      <div style="font-size: 11px; color: #f9e2af; margin-bottom: 4px;">▲ Direct IP</div>
      <pre style="background: #11111b; padding: 8px; border-radius: 6px; font-size: 11px; font-family: monospace; overflow-x: auto; margin: 0; color: #cdd6f4; border: 1px solid #313244; white-space: pre-wrap; word-break: break-all;">.\\run_tagger.ps1 -Client -HostIP \"{ip}\" -Path \"C:\\Images\" -Organize</pre>
    </div>
    <div style="background: #181825; padding: 12px; border-radius: 8px; border: 1px solid #313244;">
      <div style="font-size: 12px; color: #f9e2af; font-weight: bold; margin-bottom: 6px;">Bash (Linux / macOS)</div>
      <div style="font-size: 11px; color: #a6e3a1; margin-bottom: 4px;">● MagicDNS (Recommended)</div>
      <pre style="background: #11111b; padding: 8px; border-radius: 6px; font-size: 11px; font-family: monospace; overflow-x: auto; margin: 0 0 10px 0; color: #cdd6f4; border: 1px solid #313244; white-space: pre-wrap; word-break: break-all;">./run_tagger.sh --client --host \"{hostname}\" -p \"/path/to/images\" --organize</pre>
      <div style="font-size: 11px; color: #f9e2af; margin-bottom: 4px;">▲ Direct IP</div>
      <pre style="background: #11111b; padding: 8px; border-radius: 6px; font-size: 11px; font-family: monospace; overflow-x: auto; margin: 0; color: #cdd6f4; border: 1px solid #313244; white-space: pre-wrap; word-break: break-all;">./run_tagger.sh --client --host \"{ip}\" -p \"/path/to/images\" --organize</pre>
    </div>
  </div>
  <div style="border-top: 1px solid #313244; padding-top: 12px; text-align: center; font-size: 12px; color: #bac2de;">
    サーバーを停止するには、左のセルの<strong>停止ボタン（■）</strong>を押してください。自動的に Tailscale からログアウトされます。
  </div>
</div>
<style>
@keyframes pulse {{
  0% {{ transform: scale(0.95); box-shadow: 0 0 0 0 rgba(166, 227, 161, 0.7); }}
  70% {{ transform: scale(1); box-shadow: 0 0 0 6px rgba(166, 227, 161, 0); }}
  100% {{ transform: scale(0.95); box-shadow: 0 0 0 0 rgba(166, 227, 161, 0); }}
}}
</style>
"""
has_gpu = False
try:
    subprocess.run("nvidia-smi", stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, shell=True, check=True)
    has_gpu = True
except Exception:
    pass
args = [sys.executable, "-u", "embed_tags_universal.py", "--mode", "server", "--port", "5000"]
if has_gpu: args.append("--gpu")
env = os.environ.copy()
nvidia_libs = []
for p in site.getsitepackages():
    nvidia_dir = os.path.join(p, "nvidia")
    if os.path.exists(nvidia_dir):
        for root, dirs, files in os.walk(nvidia_dir):
            if "lib" in dirs: nvidia_libs.append(os.path.join(root, "lib"))
cuda_paths = ["/usr/local/cuda/lib64", "/usr/local/nvidia/lib64"] + nvidia_libs
env["LD_LIBRARY_PATH"] = ":".join(cuda_paths) + ":" + env.get("LD_LIBRARY_PATH", "")
process = await asyncio.create_subprocess_exec(*args, env=env, stdout=asyncio.subprocess.PIPE, stderr=asyncio.subprocess.STDOUT)
display(HTML(html_code))
try:
    while True:
        line = await process.stdout.readline()
        if not line:
            break
        sys.stdout.write(line.decode("utf-8"))
        sys.stdout.flush()
except BaseException as e:
    print(f"\n[INFO] 終了シグナルを受信しました ({type(e).__name__})。クリーンアップを実行します...")
    try:
        process.terminate()
    except Exception:
        pass
finally:
    await process.wait()
    subprocess.run(["tailscale", "logout"], capture_output=True)
    print("[INFO] Tailscale からログアウトし、安全に終了しました。")

## 4. 終了処理（Tailscaleの切断）

※上の起動セルの実行を終了した（またはボタンを押した）段階で、自動的・安全にログアウト処理は実行されるため、通常は手動で以下のセルを実行する必要はありません。何らかの理由で強制的にログアウトしたい場合のみ実行してください。

In [ ]:
!tailscale logout